# Jev API: zero-shot intent classification

Jev (TypeSafe AI) is a ready-made classification API. Unlike our other models it is
**never trained on our data**: it only receives the 14 intent names and descriptions as
options and picks one. This makes it our "new" method.

- **train**: only read by us to write the intent descriptions
- **validation**: used to choose between two option designs
- **test** and **MINDS-14**: used once each, for the final scores


### Available models
We list the models our key can use. We use `jev-latest`, the stable release
(`jev-preview` is a test version that can change at any time).


In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv("../.env")
API_KEY = os.environ["TYPESAFE_API_KEY"]
HEADERS = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

response = requests.get("https://api.typesafe.ai/v1/models", headers=HEADERS)
print(response.status_code)
print(response.json())

200
{'models': [{'name': 'jev-latest', 'description': "The latest iteration of TypeSafe's System One Model: Jev", 'release_date': '2026-09-10T18:38:01.391457+00:00'}, {'name': 'jev-preview', 'description': 'A preview version of `jev-latest`: should be better in most ways', 'release_date': '2026-09-10T18:39:06.057655+00:00'}]}


### First test: one message, three options
A single request to see the response format: the chosen option (`choice`), Jev's
`confidence`, the `probabilities` for all options, and token `usage`. The answer shows
that `jev-latest` currently points to version **jev-1.13.0**.


In [2]:
body = {
    "model": "jev-latest",
    "state": "i lost my card can you block it",
    "questions": {
        "intent": {
            "type": "choice",
            "instructions": "Which banking request is the customer making?",
            "criteria": {
                "freeze": "The customer wants to block or freeze their card or account",
                "balance": "The customer asks how much money is in their account",
                "address": "The customer wants to change their address",
            },
        }
    },
}

response = requests.post("https://api.typesafe.ai/v1/systemone", headers=HEADERS, json=body)
print(response.status_code)
print(response.json())


200
{'model': 'jev-1.13.0', 'answers': {'intent': {'type': 'choice', 'choice': 'freeze', 'confidence': 1.0, 'probabilities': {'freeze': 1.0, 'balance': 0.0, 'address': 0.0}}}, 'usage': {'input_tokens': 353, 'output_tokens': 38}}


## 1. Setup
Load the data splits and the list of 14 intents.


In [3]:
import sys 
import time 
import json
from pathlib import Path 

import pandas as pd 

sys.path.insert(0, "..")
from sklearn.metrics import f1_score
from src.evaluate import evaluate

SEED = 42 
API_URL = "https://api.typesafe.ai/v1/systemone"
MODEL = "jev-latest"


train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df =  pd.read_csv("../data/processed/test.csv")
minds_df =  pd.read_csv("../data/processed/minds14_en.csv")


INTENTS = sorted(train_df["intent"].unique())
print(len(INTENTS), "intents")
print("val:", len(val_df), "| test:", len(test_df), "| minds14:", len(minds_df))


14 intents
val: 451 | test: 452 | minds14: 1809


## 2. Two option designs

Jev only knows what we tell it about each intent, so the option text is the only thing
we can tune. We compare two designs on the validation set:
- **A: names only** (the intent name as plain words)
- **B: names + descriptions** (one sentence explaining what each intent covers)


In [4]:
# Design A: just the intent name in plain words
criteria_a = {intent: intent.replace("_", " ") for intent in INTENTS}

# Design B: a one-sentence description of each intent
criteria_b = {
    "abroad": "The customer is travelling or is in another country and has a problem or question about using their card, account or cash abroad.",
    "address": "The customer wants to change or update their home or postal address.",
    "app_error": "The mobile banking app or online banking is not working, crashing or showing an error.",
    "atm_limit": "The customer asks about or wants to change how much cash they can withdraw from an ATM.",
    "balance": "The customer wants to know how much money is in their account or whether it is overdrawn.",
    "business_loan": "The customer asks about a loan or financing for their business.",
    "card_issues": "The customer's card is lost, stolen, damaged, not arrived or not working when paying.",
    "cash_deposit": "The customer wants to deposit cash into their account.",
    "direct_debit": "The customer wants to set up, change or cancel a direct debit or regular automatic payment.",
    "freeze": "The customer wants to freeze, block, unfreeze or unblock their card or account.",
    "high_value_payment": "The customer wants to make or has a problem with a large payment or transfer.",
    "joint_account": "The customer wants to open, change or ask about a joint account shared with another person.",
    "latest_transactions": "The customer wants to see or check their recent transactions or a specific charge.",
    "pay_bill": "The customer wants to pay a bill.",
}
assert set(criteria_b) == set(INTENTS)


## 3. Classify one message

One request per message. If the API is busy (status 429) or has a temporary error (5xx),
we wait and try again, up to 5 times.


In [5]:
def classify(text, criteria, max_retries=5):
    """Send one message to jev. Returns choice, probabilities, latency, and tokens."""
    body = {
        "model": MODEL, 
        "state": text, 
        "questions": {
            "intent": {
                "type": "choice",
                "instructions": "Which banking request is the customer making?",
                "criteria": criteria,
            }
        },
    }


    for attempt in range(max_retries):
        start = time.perf_counter()
        response = requests.post(API_URL, headers=HEADERS, json=body, timeout=30)
        latency = time.perf_counter() - start

        if response.status_code == 200:
            data = response.json()
            answer = data["answers"]["intent"]
            return {
                "pred": answer["choice"],
                "confidence": answer["confidence"],
                "probabilities": answer["probabilities"],
                "latency_s": latency,
                "input_tokens": data["usage"]["input_tokens"],
                "model_version": data["model"],
            }

        if response.status_code == 429 or response.status_code >= 500:
            time.sleep(2 ** attempt)    # wait 1, 2, 4, 8, 16 seconds
            continue
        raise RuntimeError(f"Jev error {response.status_code}: {response.text}")
    raise RuntimeError("Jev did not answer after several retries")

## 4. Classify many messages

Results are saved to a file after every message. If the run stops (internet, rate limit),
running the cell again continues where it stopped instead of paying for the same messages twice.


In [6]:
RAW_DIR = Path("../results/jev/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

def run_jev(df, criteria, run_name):
    """Classify every message in df. Saves each answer as one line in a .json1 file."""
    path = RAW_DIR / f"{run_name}.jsonl"
    done =  set()
    if path.exists():
        with open(path, encoding="utf-8") as f:
            done = {json.loads(line)["row"] for line in f}

    with open(path, "a", encoding="utf-8") as f:
        for row, text in enumerate(df["text"]):
            if row in done: 
                continue
            result =  classify(text, criteria)
            result["row"] = row
            f.write(json.dumps(result) + "\n")
            time.sleep(0.1)     #small pause between requests

    results = pd.read_json(path, lines=True).sort_values("row").reset_index(drop=True)
    assert len(results) == len(df), "Not all messages were classified"
    return results


## 5. Trial run

Before running hundreds of requests, check that everything works on 20 validation messages.


In [7]:
trial = val_df.head(20)
trial_results = run_jev(trial, criteria_b, "trial_b")

trial_results["intent"] = trial["intent"].values
print("Correct:", (trial_results["pred"] == trial_results["intent"]).sum(), "of", len(trial))
print("Mean latency:", round(trial_results["latency_s"].mean(), 3), "s")
print("Mean input tokens:", round(trial_results["input_tokens"].mean()))
trial_results[["intent", "pred", "confidence"]]


Correct: 20 of 20
Mean latency: 0.472 s
Mean input tokens: 699


,intent,pred,confidence
0,abroad,abroad,1.00
1,business_loan,business_loan,1.00
2,cash_deposit,cash_deposit,1.00
3,app_error,app_error,1.00
4,business_loan,business_loan,1.00
5,high_value_payment,high_value_payment,0.98
6,pay_bill,pay_bill,1.00
7,address,address,0.95
8,direct_debit,direct_debit,1.00
9,cash_deposit,cash_deposit,0.50


## 6. Choose the option design on the validation set

We run both designs on all 451 validation messages and keep the one with the higher
macro-F1. This is the only "tuning" Jev gets. The test sets are not used here.


In [8]:
val_a = run_jev(val_df, criteria_a, "val_a")
val_b = run_jev(val_df, criteria_b, "val_b")

design_scores = pd.DataFrame([
    {"design": "A: names only",
     "val_accuracy": (val_a["pred"] == val_df["intent"]).mean(),
     "val_macro_f1": f1_score(val_df["intent"], val_a["pred"], average="macro")},
    {"design": "B: names + descriptions",
     "val_accuracy": (val_b["pred"] == val_df["intent"]).mean(),
     "val_macro_f1": f1_score(val_df["intent"], val_b["pred"], average="macro")},
]).round(4)
display(design_scores)

best_criteria = criteria_b if design_scores.loc[1, "val_macro_f1"] >= design_scores.loc[0, "val_macro_f1"] else criteria_a
best_name = "B" if best_criteria is criteria_b else "A"
print("Chosen design:", best_name)


,design,val_accuracy,val_macro_f1
0,A: names only,0.9557,0.9564
1,B: names + descriptions,0.9889,0.9890


Chosen design: B


### Design C: corrected descriptions

The validation errors pointed us to two incomplete descriptions. Checking the training
data confirmed it: 52 of the 150 `address` messages are about changing a **phone number
or email**, not a postal address, and `abroad` also covers **foreign-currency fees**.
Design C is design B with these two descriptions corrected. We check it on validation
like the others.


In [9]:
criteria_c = dict(criteria_b)
criteria_c["address"] = "The customer wants to change or update their contact details: home or postal address, phone number or email address."
criteria_c["abroad"] = "The customer is travelling or in another country, or is charged in a foreign currency, and has a problem or question about using their card, account or cash abroad."

val_c = run_jev(val_df, criteria_c, "val_c")
print("C val accuracy:", round((val_c["pred"] == val_df["intent"]).mean(), 4))
print("C val macro-F1:", round(f1_score(val_df["intent"], val_c["pred"], average="macro"), 4))


C val accuracy: 0.9933
C val macro-F1: 0.9934


## 7. Final tests

Design C had the best validation macro-F1 (0.9934), so it is used, unchanged, for both
test sets. Each test set is sent once. Predictions go to `results/predictions/` (the shared
format for the benchmark), and the full answers with probabilities go to `results/jev/`.


In [10]:
FINAL_CRITERIA = criteria_c

# Save the exact descriptions used, so the run can be reproduced
with open("../results/jev/final_criteria.json", "w", encoding="utf-8") as f:
    json.dump(FINAL_CRITERIA, f, indent=2)

def save_results(df, results, dataset_name):
    """Score with evaluate() and save predictions + probabilities."""
    metrics = evaluate(df["intent"], results["pred"], "jev", dataset_name, out_dir="../results")

    # Shared format for the benchmark: exactly text, intent, pred
    pd.DataFrame({"text": df["text"].values, "intent": df["intent"].values, "pred": results["pred"].values}
                 ).to_csv(f"../results/predictions/jev_{dataset_name}.csv", index=False)

    # Full answers with one probability column per intent, kept separately
    probs = pd.DataFrame(results["probabilities"].tolist())[INTENTS].add_prefix("prob_")
    details = pd.concat([df[["text", "intent"]].reset_index(drop=True),
                         results[["pred", "confidence", "latency_s", "input_tokens"]], probs], axis=1)
    details.to_csv(f"../results/jev/jev_probabilities_{dataset_name}.csv", index=False)
    return metrics


In [11]:
test_results = run_jev(test_df, FINAL_CRITERIA, "test_c")
save_results(test_df, test_results, "synthetic_test")


=== jev on synthetic_test ===
Accuracy: 0.9889   Macro-F1: 0.9890

                     precision    recall  f1-score   support

             abroad       1.00      0.97      0.98        32
            address       1.00      1.00      1.00        32
          app_error       1.00      0.97      0.98        33
          atm_limit       1.00      1.00      1.00        32
            balance       1.00      1.00      1.00        32
      business_loan       1.00      1.00      1.00        33
        card_issues       0.91      1.00      0.96        32
       cash_deposit       1.00      1.00      1.00        33
       direct_debit       0.97      1.00      0.98        32
             freeze       0.97      0.94      0.95        32
 high_value_payment       1.00      1.00      1.00        32
      joint_account       1.00      1.00      1.00        32
latest_transactions       1.00      1.00      1.00        32
           pay_bill       1.00      0.97      0.98        33

           accur

{'model': 'jev',
 'dataset': 'synthetic_test',
 'accuracy': 0.9889,
 'macro_f1': 0.989}

In [12]:
minds_results = run_jev(minds_df, FINAL_CRITERIA, "minds14_c")
save_results(minds_df, minds_results, "minds14")


=== jev on minds14 ===
Accuracy: 0.9508   Macro-F1: 0.9510

                     precision    recall  f1-score   support

             abroad       1.00      0.95      0.97       115
            address       0.94      1.00      0.97       131
          app_error       0.95      0.94      0.95       125
          atm_limit       1.00      0.98      0.99       137
            balance       0.94      1.00      0.97       131
      business_loan       0.99      0.92      0.96       130
        card_issues       0.90      0.94      0.92       138
       cash_deposit       0.97      0.85      0.91       135
       direct_debit       1.00      0.94      0.97       125
             freeze       0.97      0.88      0.92       128
 high_value_payment       0.85      0.94      0.89       126
      joint_account       0.98      1.00      0.99       129
latest_transactions       0.91      0.98      0.94       129
           pay_bill       0.95      0.98      0.97       130

           accuracy    

{'model': 'jev', 'dataset': 'minds14', 'accuracy': 0.9508, 'macro_f1': 0.951}

## 8. Analysis

### 8.1 Comparison with the other models


In [13]:
metrics = pd.read_csv("../results/metrics.csv")
comparison = metrics.pivot(index="model", columns="dataset", values="macro_f1")
comparison["drop"] = comparison["synthetic_test"] - comparison["minds14"]
comparison.sort_values("minds14", ascending=False).round(4)


dataset,minds14,synthetic_test,drop
model,,,
jev,0.951,0.9890,0.0380
sbert_svm,0.917,0.9866,0.0696
tfidf_lr,0.864,0.9386,0.0746


### 8.2 Is the difference real? (paired bootstrap)

Jev is compared with the best trained model, `sbert_svm`, on the same messages.
If the 95% interval of the difference does not include 0, the difference is unlikely to be chance.


In [14]:
from src.bootstrap import bootstrap_ci, paired_bootstrap_diff

rows = []
for ds in ["synthetic_test", "minds14"]:
    jev = pd.read_csv(f"../results/predictions/jev_{ds}.csv")
    sbert = pd.read_csv(f"../results/predictions/sbert_svm_{ds}.csv")
    low, high = paired_bootstrap_diff(jev["intent"].values, jev["pred"].values, sbert["pred"].values)
    rows.append({"dataset": ds, "diff_ci_low": round(low, 4), "diff_ci_high": round(high, 4),
                 "clear_difference": low > 0 or high < 0})
pd.DataFrame(rows)


,dataset,diff_ci_low,diff_ci_high,clear_difference
0,synthetic_test,-0.0102,0.0170,False
1,minds14,0.0236,0.0456,True


### 8.3 Errors on MINDS-14
The most common confusions, with example messages.


In [15]:
minds_details = pd.read_csv("../results/jev/jev_probabilities_minds14.csv")
errors = minds_details[minds_details["pred"] != minds_details["intent"]]
print("Errors:", len(errors), "of", len(minds_details))
display(errors.groupby(["intent", "pred"]).size().sort_values(ascending=False).head(8).rename("count").to_frame())

pd.set_option("display.max_colwidth", 120)
for true, pred in [("cash_deposit", "high_value_payment"), ("freeze", "card_issues"), ("business_loan", "address")]:
    print(f"\n{true} -> {pred}")
    display(errors[(errors["intent"] == true) & (errors["pred"] == pred)][["text", "confidence"]].head(4))


Errors: 89 of 1809


,,count
intent,pred,
cash_deposit,high_value_payment,17
freeze,card_issues,9
business_loan,address,5
card_issues,app_error,4
freeze,latest_transactions,4
high_value_payment,pay_bill,4
direct_debit,latest_transactions,4
app_error,balance,3



cash_deposit -> high_value_payment


,text,confidence
430,transfer money to the account,0.99
442,how do I transfer money to a different account,0.99
474,I'm using the bank can you help me transfer money into my account thank you,0.90
1112,hi I'm just wondering how I transfer money to my account please,0.95



freeze -> card_issues


,text,confidence
126,hi I've lost my bank card and please stop all transactions I don't want anybody using my card for anything I don't k...,0.82
139,see the security hold put on my card,0.30
145,hello GIF I'm going cuz my card got stolen I would like to stop all transactions for my car completely,0.68
157,there has been an issue with my card and I would like to breathe,0.99



business_loan -> address


,text,confidence
542,I'm calling about a news this thing business phone,0.73
551,I need information about a business phone,0.86
941,business phone,0.90
960,hello I'd like to enquire about a new business phone it's for an existing business,0.72


### 8.4 Does confidence tell us when Jev is wrong?


In [16]:
for name, details in [("synthetic_test", pd.read_csv("../results/jev/jev_probabilities_synthetic_test.csv")),
                      ("minds14", minds_details)]:
    high = details[details["confidence"] >= 0.9]
    low = details[details["confidence"] < 0.9]
    print(f"{name}: confidence >= 0.9 -> {len(high)} messages, accuracy {(high['pred'] == high['intent']).mean():.4f} | "
          f"confidence < 0.9 -> {len(low)} messages, accuracy {(low['pred'] == low['intent']).mean():.4f}")


synthetic_test: confidence >= 0.9 -> 427 messages, accuracy 1.0000 | confidence < 0.9 -> 25 messages, accuracy 0.8000
minds14: confidence >= 0.9 -> 1654 messages, accuracy 0.9837 | confidence < 0.9 -> 155 messages, accuracy 0.6000


### 8.5 Cost and latency
Counted over **all** requests in this notebook (trial, three designs on validation, both test sets).
Price: $0.04 per million input tokens (TypeSafe pay-as-you-go price at the time of the experiment).


In [17]:
all_runs = pd.concat([pd.read_json(p, lines=True).assign(run=p.stem) for p in RAW_DIR.glob("*.jsonl")])
summary = all_runs.groupby("run").agg(requests=("pred", "size"),
                                      mean_latency_s=("latency_s", "mean"),
                                      mean_input_tokens=("input_tokens", "mean")).round(3)
display(summary)

total_tokens = all_runs["input_tokens"].sum()
print("Total requests:", len(all_runs))
print("Total input tokens:", total_tokens)
print(f"Estimated total cost: ${total_tokens / 1_000_000 * 0.04:.2f}")
print("Model versions used:", all_runs["model_version"].unique())


,requests,mean_latency_s,mean_input_tokens
run,,,
minds14_c,1809,0.460,718.940
test_c,452,0.460,717.131
trial_b,20,0.472,699.350
val_a,451,0.452,499.399
val_b,451,0.455,700.399
val_c,451,0.453,717.399


Total requests: 3634
Total input tokens: 2503348
Estimated total cost: $0.10
Model versions used: <StringArray>
['jev-1.13.0']
Length: 1, dtype: str


## 9. Conclusion

**Setup.** `jev-latest` (version `jev-1.13.0` for every request), zero-shot, with design C:
one-sentence descriptions of the 14 intents, corrected for `address` and `abroad` after
checking the training data. Design C was chosen on validation (macro-F1 0.9934, vs 0.9890 for
B and 0.9564 for names only). Jev was never trained on our data.

**Results.**
- Synthetic test: macro-F1 **0.989**. Statistically tied with `sbert_svm` (0.987); the
  synthetic test set is too easy to separate the best models.
- MINDS-14 (real calls): macro-F1 **0.951**, clearly better than `sbert_svm` (0.917) and
  `tfidf_lr` (0.864). The paired bootstrap interval of the difference to `sbert_svm` excludes 0.
- Jev drops only about **4 points** from synthetic to real data; the trained models drop about **7**.
  A likely reason: the trained models partly learned the style of our synthetic messages,
  while Jev only relies on the meaning of the intent descriptions.

**Option descriptions matter.** Descriptions cut validation errors from 20 (names only) to 5,
mainly by separating `freeze` and `abroad` from `card_issues`. For a zero-shot model,
writing the descriptions is the equivalent of tuning.

**Main errors on MINDS-14.**
- `cash_deposit` -> `high_value_payment` (17x): many MINDS-14 `cash_deposit` calls are about
  *transferring money into an account*, not depositing cash. Our description ("deposit cash")
  does not cover this, so the intent is defined more broadly in MINDS-14 than in our data.
- `freeze` -> `card_issues` (9x): a lost or stolen card plus "stop all transactions" fits both intents.
- `business_loan` -> `address` (5x): the speech recognition turned "business loan" into
  "business phone", and our corrected `address` description mentions phone numbers.
  A side effect of design C on noisy transcripts.

**Confidence is useful.** On MINDS-14, the 91% of messages with confidence >= 0.9 are 98.4%
correct, while the rest are only 60% correct. In practice, low-confidence messages could be
passed to a human agent.

**Cost and speed.** About 720 input tokens and 0.46 s per message. All 2.5 million tokens used
in this notebook cost about $0.10.

**Limitations.**
- Design C was created after looking at validation errors (the test sets were not used).
- `jev-latest` is an alias; later versions may give different results.
- The synthetic test set is near its ceiling, so only MINDS-14 separates the best models.
